# Notebook 03 — QLoRA Fine-tuning of the Generator (Qwen2.5-7B-Instruct)

**Pipeline position:** experimental component (ablation row **C4**). NOT enabled in `--variant prod`.

Self-contained QLoRA run:
1. Build instruction data `(system, "Soru: ...", answer)` from the Turkish legal QA datasets (+ refusal examples). No retrieved context -- single-passage format.
2. Load Qwen2.5-7B-Instruct in 4-bit NF4, attach LoRA (r=32, alpha=64) on all attention+MLP projections.
3. Train 1 epoch with paged_adamw_8bit, save the adapter, smoke-test.

**Documented negative result:** the single-passage training format does not transfer to
multi-passage RAG inference (-0.065 Token F1), though faithfulness rose +24pp. Production
therefore uses the base generator. See report section on negative results.

> Recovered from the original Colab training session.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check if QLoRA v2 saved any checkpoints
import os
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/hukuk-rag')

# Check v2 checkpoints
v2_path = DRIVE / 'models' / 'qwen-qlora-v2' / 'checkpoints'
print("=== QLoRA v2 ===")
if v2_path.exists():
    for item in sorted(v2_path.iterdir()):
        if item.is_dir():
            files = list(item.iterdir())
            print(f"  {item.name}/ ({len(files)} files)")
        else:
            print(f"  {item.name}")
else:
    print("  NOT FOUND — no v2 checkpoints saved")

# Also check v1 (the broken one)
v1_path = DRIVE / 'models' / 'qwen-qlora' / 'checkpoints'
print("\n=== QLoRA v1 (broken) ===")
if v1_path.exists():
    for item in sorted(v1_path.iterdir()):
        if item.is_dir():
            print(f"  {item.name}/")
        else:
            print(f"  {item.name}")

In [ ]:
############################################################
# QLoRA v2 — FULL SELF-CONTAINED CELL
# Install → Load → Train → Save → Test
############################################################
!pip install -q trl peft bitsandbytes accelerate datasets transformers

import torch, gc, json, random
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

DRIVE = Path('/content/drive/MyDrive/hukuk-rag')
QLORA_DIR = DRIVE / 'models' / 'qwen-qlora-v2'
QLORA_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM = (
    "Sen bir Türk hukuku uzmanısın. Soruyu kısa ve öz şekilde yanıtla (2-3 cümle). "
    "İlgili kanun maddelerine atıfta bulun. Bilgi yoksa 'Bu konuda yeterli bilgi bulunamadı' de."
)

# ── Dataset: raw messages only ──
print("Building dataset...")
qa1 = pd.read_parquet(str(DRIVE / 'data' / 'raw' / 'turkish_law_qa.parquet'))
qa2 = pd.read_parquet(str(DRIVE / 'data' / 'raw' / 'turkish_law_chatbot.parquet'))
qa1 = qa1.rename(columns={'question': 'query', 'answer': 'answer'})
qa2 = qa2.rename(columns={'Soru': 'query', 'Cevap': 'answer'})
qa_all = pd.concat([qa1[['query','answer']], qa2[['query','answer']]], ignore_index=True).dropna().reset_index(drop=True)
qa_sample = qa_all.sample(min(10000, len(qa_all)), random_state=42).reset_index(drop=True)

training_examples = []
for _, row in qa_sample.iterrows():
    training_examples.append({"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Soru: {row['query']}"},
        {"role": "assistant", "content": row['answer']},
    ]})

unanswerable = ["Bu konuda verilen bağlamda yeterli bilgi bulunmamaktadır.",
    "Bu soruya yanıt verilebilmesi için ek bilgiye ihtiyaç vardır."]
for i in range(300):
    training_examples.append({"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Soru: Hayali soru #{i}"},
        {"role": "assistant", "content": random.choice(unanswerable)},
    ]})
random.shuffle(training_examples)
dataset = Dataset.from_list(training_examples)
print(f"Dataset: {len(dataset)} examples, columns: {dataset.column_names}")
del qa1, qa2, qa_all, qa_sample; gc.collect()

# ── Load model ──
print("Loading Qwen2.5-7B-Instruct...")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM"))
model.print_trainable_parameters()

# ── Train ──
training_args = SFTConfig(
    output_dir=str(QLORA_DIR / 'checkpoints'),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    warmup_steps=30,
    weight_decay=0.01,
    bf16=True,
    logging_steps=25,
    save_steps=200,
    save_total_limit=2,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    report_to="none",
    optim="paged_adamw_8bit",
    max_length=2048,
)

trainer = SFTTrainer(model=model, args=training_args,
    train_dataset=dataset, processing_class=tokenizer)

steps = len(dataset) // (2 * 16)
print(f"\nTraining: {steps} steps, 1 epoch")
trainer.train()

# ── Save ──
print("\nSaving adapter...")
trainer.model.save_pretrained(str(QLORA_DIR / 'adapter'))
tokenizer.save_pretrained(str(QLORA_DIR / 'adapter'))
print(f"Saved to {QLORA_DIR / 'adapter'}")

# ── Quick test ──
print("\nTesting...")
test_q = "Kasten adam öldürme suçunun cezası nedir?"
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"Soru: {test_q}"}]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
with torch.inference_mode():
    out = trainer.model.generate(**inputs, max_new_tokens=200, temperature=0.1,
        do_sample=True, pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Q: {test_q}")
print(f"A: {answer[:400]}")
print("\nDONE.")

In [ ]:
############################################################
# FIX: Disable gradient checkpointing + enable cache for inference
############################################################
trainer.model.gradient_checkpointing_disable()
trainer.model.config.use_cache = True
trainer.model.eval()

# Test again
test_q = "Kasten adam öldürme suçunun cezası nedir?"
msgs = [
    {"role": "system", "content": "Sen bir Türk hukuku uzmanısın. Soruyu kısa ve öz şekilde yanıtla (2-3 cümle). İlgili kanun maddelerine atıfta bulun."},
    {"role": "user", "content": f"Soru: {test_q}"}
]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)

with torch.inference_mode():
    out = trainer.model.generate(
        **inputs, max_new_tokens=200, temperature=0.1,
        do_sample=True, pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2,  # penalize repetition
    )
answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"Q: {test_q}")
print(f"A: {answer[:400]}")
print(f"Length: {len(answer)} chars")

In [ ]:
# Test a few more questions
test_qs = [
    "Boşanma davasında nafaka nasıl belirlenir?",
    "İdari yargıda dava açma süresi ne kadardır?",
    "Anonim şirketlerde yönetim kurulu üyelerinin sorumluluğu nedir?",
]

for q in test_qs:
    msgs = [
        {"role": "system", "content": "Sen bir Türk hukuku uzmanısın. Soruyu kısa ve öz şekilde yanıtla (2-3 cümle). İlgili kanun maddelerine atıfta bulun."},
        {"role": "user", "content": f"Soru: {q}"}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    with torch.inference_mode():
        out = trainer.model.generate(**inputs, max_new_tokens=200, temperature=0.1,
            do_sample=True, pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.2)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\nQ: {q}")
    print(f"A: {answer[:300]}")

print("\nQLoRA v2 model is WORKING.")